# Deliverable 3: analysis of the substitution scores

Sarthak Aggarwal (MT26139) and Divyanshu Joshi (MT26120)

This notebook looks at how the flavor-overlap scores behave: how many of our 250 ingredients have real flavor compounds, how Jaccard and IDF-weighted overlap differ, and how they do on the category check from `D3_7_evaluate.py`. Run `python D3_7_evaluate.py` first so the result files are up to date, then run all cells from the top.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, os.getcwd())
from D3_5_substitute import Substituter
from D3_7_evaluate import SUMMARY_CSV, pair_score_gap

BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#898884"      # Jaccard, IDF, chance
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e3e2de"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlesize": 11,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True})

sub = Substituter()
mol, tags = sub.molecules, sub.tags
print(len(sub.rows), "ingredients loaded")

## 1. How much of the 250 has real flavor compounds?
An ingredient counts as "molecular" when it links to an Ahn ingredient with at least 5 compounds. The rest fall back to the smell tags from Deliverable 1, which is the weaker score.

In [ ]:
n_mol = len(mol.compounds)
n_tag = sum(1 for i in sub.rows if i not in mol.compounds and tags.has(i))
n_none = len(sub.rows) - n_mol - n_tag
labels = ["No data", "Tags only: smell + rasa (weaker)", "Flavor compounds (Ahn)"]
values = [n_none, n_tag, n_mol]
fig, ax = plt.subplots(figsize=(7, 2.4))
ax.barh(labels, values, color=[GREY, ORANGE, BLUE], height=0.55)
for y, v in enumerate(values):
    ax.text(v + 2, y, "%d (%.0f%%)" % (v, 100 * v / len(sub.rows)), va="center", color=INK)
ax.set_xlim(0, max(values) * 1.2)
ax.set_xlabel("ingredients out of %d" % len(sub.rows))
ax.set_title("Where each ingredient's flavor data comes from")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

## 2. Jaccard against IDF: how different are they?
For every ingredient with compounds we take the best substitute under each method. First, how high do the best scores reach? Second, how much do the top 5 lists agree?

In [ ]:
top = {"jaccard": [], "idf": []}
five = {"jaccard": {}, "idf": {}}
for q in mol.compounds:
    for m in top:
        res = sub.substitutes(q, top_k=5, method=m, basis="molecular")
        if res:
            top[m].append(res[0]["score"])
        five[m][q] = [r["index"] for r in res]

fig, ax = plt.subplots(figsize=(7, 3.4))
for m, color, name in (("jaccard", BLUE, "Jaccard"), ("idf", ORANGE, "IDF-weighted")):
    xs = sorted(top[m])
    ys = [(k + 1) / len(xs) for k in range(len(xs))]
    ax.plot(xs, ys, color=color, linewidth=2, label=name)
ax.set_xlabel("score of the best substitute")
ax.set_ylabel("share of ingredients at or below")
ax.set_title("Score of the best substitute, by method")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
print("median best score: jaccard %.3f, idf %.3f" % (pd.Series(top["jaccard"]).median(), pd.Series(top["idf"]).median()))

In [ ]:
agree = pd.Series([len(set(five["jaccard"][q]) & set(five["idf"][q])) for q in mol.compounds])
counts = agree.value_counts().reindex(range(0, 6), fill_value=0)
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.bar(counts.index, counts.values, color=BLUE, width=0.6)
for x, v in zip(counts.index, counts.values):
    ax.text(x, v + 1, str(v), ha="center", color=INK)
ax.set_ylim(0, counts.max() * 1.12)
ax.set_xlabel("how many of the top 5 substitutes the two methods share")
ax.set_ylabel("ingredients")
ax.set_title("The two methods mostly pick the same substitutes")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()
print("mean overlap: %.2f of 5" % agree.mean())

## 3. Category check
No hand-made list of swaps yet, so this uses a rough stand-in: does a top-ranked substitute belong to the same Ahn food category (fruit, spice, dairy, ...)? The grey bars show what picking at random would give. Numbers are read from `D3_7_evaluation_summary.csv`.

In [ ]:
summary = pd.read_csv(SUMMARY_CSV)
cat = summary[(summary.evaluation == "category_check") & (summary.split == "all")].set_index("method")
ks = ["hit1", "hit3", "hit5"]
fig, ax = plt.subplots(figsize=(7, 3.4))
width = 0.26
for pos, (m, color, name) in enumerate((("jaccard", BLUE, "Jaccard"), ("idf", ORANGE, "IDF-weighted"), ("random", GREY, "Random order"))):
    xs = [k + (pos - 1) * width for k in range(3)]
    vals = [cat.loc[m, k] for k in ks]
    ax.bar(xs, vals, width=width * 0.9, color=color, label=name)
    for x, v in zip(xs, vals):
        ax.text(x, v + 0.012, "%.2f" % v, ha="center", color=INK, fontsize=8)
ax.set_xticks(range(3))
ax.set_xticklabels(["Hit@1", "Hit@3", "Hit@5"])
ax.set_ylim(0, 1)
ax.set_ylabel("share of ingredients")
ax.set_title("A top substitute shares the food category far more often than chance")
ax.legend(frameon=False, loc="upper left", ncol=3)
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()
summary[summary.evaluation == "category_check"]

In [ ]:
gap = pair_score_gap(sub)
pd.DataFrame(gap).T.astype({"pairs": int})

The table shows the mean score of ingredient pairs in the same Ahn category and in different categories. A useful score should be clearly higher for the first group.

## 4. Gold pairs (known Indian swaps)
Filled in once `D3_6_gold_pairs.csv` has rows and `D3_7_evaluate.py` has been run again.

In [ ]:
gold = summary[summary.evaluation == "gold_pairs"]
if gold.empty:
    print("No gold pairs yet: add rows to D3_6_gold_pairs.csv and run D3_7_evaluate.py again.")
else:
    display(gold)

## 5. Examples
Top 5 substitutes under both methods.

In [ ]:
by_name = {r["english_name"]: i for i, r in sub.rows.items()}
rows = []
for name in ["cumin seeds", "tomato", "mango", "cottage cheese", "onion", "coriander seeds"]:
    for m in ("jaccard", "idf"):
        res = sub.substitutes(by_name[name], top_k=5, method=m)
        rows.append({"ingredient": name, "method": m,
                     "top 5": ", ".join("%s (%.0f%%)" % (r["english_name"], r["percent"]) for r in res)})
pd.set_option("display.max_colwidth", 140)
pd.DataFrame(rows)

## 6. Where it is weak
Ingredients whose best compound score is low, and tag-based queries where many substitutes tie.

In [ ]:
weak = []
for q in mol.compounds:
    res = sub.substitutes(q, top_k=1, method="jaccard", basis="molecular")
    weak.append({"ingredient": sub.rows[q]["english_name"], "compounds": len(mol.compounds[q]),
                 "best score": res[0]["score"] if res else 0.0, "best substitute": res[0]["english_name"] if res else ""})
pd.DataFrame(weak).sort_values("best score").head(10).reset_index(drop=True)

In [ ]:
ties = []
for q in sub.rows:
    if mol.has(q) or not tags.has(q):
        continue
    res = sub.substitutes(q, top_k=10 ** 6, basis="tags")
    if res:
        ties.append({"ingredient": sub.rows[q]["english_name"], "best score": res[0]["score"],
                     "substitutes tied at the best score": sum(1 for r in res if r["score"] == res[0]["score"])})
ties = pd.DataFrame(ties)
print("tag-based queries:", len(ties), "| median number tied at the top:", int(ties.iloc[:, 2].median()))
ties.sort_values(ties.columns[2], ascending=False).head(8).reset_index(drop=True)

## 7. Compounds against tags, and who wins
`Substituter` now scores every pair two ways (compounds, and smell + rasa tags) and the higher score wins. First, the category check for each basis on the same 141 ingredients. Second, how often each side wins. Third, the review sheet `D3_7_win_review.csv`: a person marks `good_win` (yes or no) for each winning substitute, and the table shows good wins per winner.

In [ ]:
bases = summary[(summary.evaluation == "category_check_bases") & (summary.split == "all")]
bases[["method", "n", "hit1", "hit3", "hit5", "mrr"]].reset_index(drop=True)

In [ ]:
wins = summary[summary.evaluation == "win_share"][["method", "n", "note"]]
display(wins)
review = summary[summary.evaluation == "win_review"]
if review.empty:
    print("No marked wins yet: fill the good_win column in D3_7_win_review.csv and run D3_7_evaluate.py again.")
else:
    display(review[["method", "n", "note"]])

## What this tells us
- Only part of the 250 has real flavor compounds. The rest use smell tags, which are short labels and often give many equal scores.
- Jaccard and IDF-weighted overlap pick nearly the same top substitutes, so the choice matters less than the data behind them.
- Both scores beat random order on the category check, but that check is rough. The real test is the list of known swaps in `D3_6_gold_pairs.csv`.